In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

MIN_MATCH_COUNT = 10
img1 = cv2.imread('mynote.jpg',0) # queryImage
img2 = cv2.imread('books.jpg',0) # trainImage

# Initiate SIFT detector
sift = cv2.SIFT_create()
# find the keypoints and descriptors with SIFT
kp1, des1 = sift.detectAndCompute(img1,None)
kp2, des2 = sift.detectAndCompute(img2,None)

FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1,des2,k=2)

# store all the good matches as per Lowe's ratio test.
good = []
for m,n in matches:
    if m.distance < 0.5*n.distance:
        good.append(m)

if len(good)>MIN_MATCH_COUNT:
    src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
    dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC,5.0)
    matchesMask = mask.ravel().tolist()
    h,w = img1.shape
    pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
    dst = cv2.perspectiveTransform(pts,M)
    img2 = cv2.polylines(img2,[np.int32(dst)],True,255,3, cv2.LINE_AA)
else:
    print("Not enough matches are found - %d/%d" % (len(good),MIN_MATCH_COUNT))
    matchesMask = None

cap = cv2.VideoCapture(0)

while(cap.isOpened()):
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kp2, des2 = sift.detectAndCompute(gray, None)

    if des2 is not None and len(kp2) > 10:
        matches = flann.knnMatch(des1, des2, k=2)
        
        good = []
        for m, n in matches:
            if m.distance < 0.6 * n.distance:
                good.append(m)

        if len(good) > MIN_MATCH_COUNT:
            src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
            dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

            M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            

            if M is not None:
                matchesMask = mask.ravel().tolist()
                h, w = img1.shape
                pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
                dst = cv2.perspectiveTransform(pts, M)
                frame = cv2.polylines(frame, [np.int32(dst)], True, (0, 255, 0), 3, cv2.LINE_AA)
            else:
                matchesMask = None
        else:
            print(f"Matches not enough: {len(good)}/{MIN_MATCH_COUNT}")
            matchesMask = None
            
        draw_params = dict(matchColor=(0, 255, 0), 
                          singlePointColor=None, 
                          matchesMask=matchesMask, 
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        
        img3 = cv2.drawMatches(img1, kp1, frame, kp2, good, None, **draw_params)
        cv2.imshow('frame', img3)
    else:
        cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Not enough matches are found - 0/10
Matches not enough: 6/10
Matches not enough: 1/10
Matches not enough: 0/10
Matches not enough: 1/10
Matches not enough: 1/10
Matches not enough: 0/10
Matches not enough: 5/10
Matches not enough: 2/10
Matches not enough: 4/10
Matches not enough: 4/10
Matches not enough: 7/10
Matches not enough: 1/10
Matches not enough: 2/10
Matches not enough: 8/10
Matches not enough: 2/10
Matches not enough: 3/10
Matches not enough: 1/10
Matches not enough: 6/10
Matches not enough: 5/10
Matches not enough: 0/10
Matches not enough: 6/10
Matches not enough: 7/10
Matches not enough: 4/10
Matches not enough: 0/10
Matches not enough: 4/10
Matches not enough: 1/10
Matches not enough: 2/10
Matches not enough: 4/10
Matches not enough: 1/10
Matches not enough: 0/10
Matches not enough: 1/10
Matches not enough: 2/10
Matches not enough: 1/10
Matches not enough: 3/10
Matches not enough: 3/10
Matches not enough: 9/10
Matches not enough: 9/10
Matches not enough: 3/10
Matches not en